## Etapa 1 — Criação do Catálogo de Dados

No Databricks, um **catálogo** (catalog) é o nível mais alto da hierarquia do Unity Catalog, funcionando como um contêiner lógico que organiza schemas (bancos de dados) e tabelas. Abaixo é criado o catálogo `mvpDataEngineer` que será utilizado em todo o projeto.

In [0]:

%sql
DROP CATALOG IF EXISTS mvpDataEngineer CASCADE

In [0]:
%sql
-- Cria o catálogo 'mvpDataEngineer' no Unity Catalog.
-- Este catálogo conterá todos os schemas e tabelas do projeto.
CREATE CATALOG IF NOT EXISTS mvpdataengineer

## Etapa 2 — Seleção do Catálogo de Trabalho

Após criar o catálogo é preciso definir que ele será o **catálogo ativo** na sessão atual. Isso significa que, a partir deste ponto, todas as operações que referenciarem apenas o nome do schema (sem qualificar com o nome do catálogo) serão resolvidas automaticamente dentro de `mvpDataEngineer`.

In [0]:
%sql
-- Define 'mvpDataEngineer' como o catálogo ativo da sessão.
-- Comandos subsequentes não precisarão qualificar o nome do catálogo.
USE CATALOG mvpDataEngineer

## Etapa 3 — Criação dos schemas (arquitetura medalhão)

A arquitetura organiza os dados em quatro camadas, cada uma com uma responsabilidade única:

| Camada | Responsabilidade | Transformação permitida |
|---|---|---|
| **staging** | Área de chegada. Isola o parse dos arquivos de origem. | Nenhuma regra de negócio. Tudo como texto. |
| **bronze** | Dado bruto persistido com metadados de ingestão e histórico de cargas. | Somente acréscimo de metadados. |
| **silver** | Dado limpo, tipado, padronizado e deduplicado. | Limpeza e tipagem, mantendo a granularidade da origem. |
| **gold** | Modelo dimensional e tabelas analíticas. | Modelagem, agregação e cálculo de métricas. |

A separação entre `staging` e `bronze` é deliberada: falhas de leitura de arquivo
(codificação, cabeçalho deslocado, planilha renomeada) ficam contidas na staging e não
contaminam o histórico preservado na bronze.

In [0]:
 %sql
CREATE SCHEMA IF NOT EXISTS staging
    COMMENT 'Camada de chegada. Cópia fiel dos arquivos baixados do Google Drive, com todas as colunas preservadas como texto e sem nenhuma regra de negócio aplicada. Contém também o Volume onde os arquivos brutos são armazenados.';
    
CREATE SCHEMA IF NOT EXISTS bronze
    COMMENT 'Dados brutos persistidos em Delta com metadados de rastreabilidade (arquivo de origem, data de ingestão e data de carga). Preserva o histórico e permite reprocessar as camadas seguintes sem voltar aos arquivos originais.';

CREATE SCHEMA IF NOT EXISTS silver
    COMMENT 'Dados limpos, tipados, padronizados e deduplicados, mantendo a granularidade da origem. Camada onde os problemas de qualidade identificados são tratados.';

CREATE SCHEMA IF NOT EXISTS gold
    COMMENT 'Modelo dimensional (star schema) e tabelas analíticas prontas para responder às perguntas de negócio do projeto.';